In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# 1. Device configuration (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set manual seeds for reproducibility on both CPU and CUDA
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# 2. Dataset sent directly to CUDA device
X = torch.tensor([[0.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 0.0],
                  [1.0, 1.0]], dtype=torch.float32, device=device)

Y = torch.tensor([[0.0],
                  [1.0],
                  [1.0],
                  [0.0]], dtype=torch.float32, device=device)

In [ ]:
# 3. Model Definition
class XORNet(nn.Module):
    def __init__(self):
        super(XORNet, self).__init__()
        self.fc1 = nn.Linear(in_features=2, out_features=2)
        self.fc2 = nn.Linear(in_features=2, out_features=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        z1 = self.fc1(x)
        a1 = self.sigmoid(z1)
        z2 = self.fc2(a1)
        a2 = self.sigmoid(z2)
        return a2

# Instantiate and move model parameters to CUDA VRAM
model = XORNet().to(device)

In [ ]:
# Other way to define model
'''
model = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1),
    nn.Sigmoid()
)
'''

In [ ]:
# TRAINING SETUP: loss and optimizer

# 4. Custom RMSE Loss
mse_criterion = nn.MSELoss()

def rmse_loss(y_pred, y_true):
    return torch.sqrt(mse_criterion(y_pred, y_true) + 1e-8)

# 5. Optimizer (SGD)
optimizer = optim.SGD(model.parameters(), lr=1.0)

In [ ]:
# 6. Training Loop
epochs = 10000

for epoch in range(1, epochs + 1):
    optimizer.zero_grad()

    predictions = model(X)
    loss = rmse_loss(predictions, Y)

    loss.backward()
    optimizer.step()

    if epoch % 2000 == 0:
        # .item() copies the scalar loss value from GPU to Python float
        print(f"Epoch {epoch:5d} | RMSE Loss: {loss.item():.6f}")

In [ ]:
# 7. Evaluation
print("\n--- Final Predictions ---")
model.eval()
with torch.no_grad():
    final_preds = model(X)

    # Move tensors back to host CPU memory and convert to NumPy for display
    X_cpu = X.cpu().numpy().astype(int)
    Y_cpu = Y.cpu().numpy().astype(int)
    preds_cpu = final_preds.cpu().numpy()

    for i in range(len(X_cpu)):
        inp = X_cpu[i]
        target = Y_cpu[i][0]
        pred = preds_cpu[i][0]
        rounded = int(round(pred))
        print(f"Input: {inp} | Target: {target} | Predicted: {pred:.4f} (Round: {rounded})")